# Real-World Conflation: Sundbyberg Road Map Matching

Welcome to the Sundbyberg map conflation matching dashboard! This notebook is configured to match and conflate your Sundbyberg road networks using local, self-contained datasets:
1. **OSM Traffic Enrichment Edges**: `data/osm_edges.csv`
2. **Sweden NVDB Standardized Directed Edges**: `data/sweden_edges.csv`

### 💡 GitHub-Ready & Portable Data
Both road networks have been pre-processed and exported as standard CSV files directly inside the project's `data/` directory. All coordinates and spatial geometries are stored as Well-Known Text (WKT). 

When you run this notebook, DuckDB Spatial will automatically parse the local WKT files into active in-memory geometry tables. This makes the entire repository fully portable, ready for GitHub, and immune to any read-write database file locking conflicts!

## Step 1: Inspect Database Schemas (Read-Only)

You can run this cell to verify all tables and columns present in both databases. We use `read_only=True` connections to prevent any file-locking conflicts with other active sessions.

In [ ]:
import pandas as pd

print("==================================================")
print("         INSPECTING LOCAL DATASETS (CSV)")
print("==================================================")

print("--> OSM Road Network (osm_edges.csv):")
df_osm = pd.read_csv("../data/osm_edges.csv")
display(df_osm.head())

print("\n--> Sweden Road Network (sweden_edges.csv):")
df_sweden = pd.read_csv("../data/sweden_edges.csv")
display(df_sweden.head())

## Step 2: Configured Table & Column Names

The tables and columns are configured as follows:
* **Source A**: OSM directed edges table is `driving.edges`.
* **Source B**: Preprocessed directed Sweden edges table is `main.vehicle_edges_directed` (generated inside Database A during Step 3).
* **Projection System**: Projected into **SWEREF99 TM (EPSG:3006)**—Sweden's national metric coordinate system.

In [ ]:
# Config from DB A (OSM)
TABLE_A = "driving_edges"
ID_A = "edge_id"
GEOM_A = "geometry"

# Config from DB B (Sweden NVDB preprocessed table)
TABLE_B = "vehicle_edges_directed"
ID_B = "directed_id"
GEOM_B = "geometry"

# Local projected coordinate system in meters for Sweden (SWEREF99 TM)
UTM_SRID = 3006

print("✅ Local table and column configurations prepared!")

## Step 3: Directed Preprocessing & Matcher Initialization

We initialize our `DuckDBMapMatcher`, dynamically construct the directed Sweden edges representation `main.vehicle_edges_directed`, configure the sources, and perform candidate generation.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from network_matching import DuckDBMapMatcher

# 1. Initialize matcher with a clean in-memory DuckDB connection
print("[Step 1] Initializing in-memory map matcher...")
matcher = DuckDBMapMatcher()

# 2. Load local WKT datasets and parse them into in-memory spatial tables
print("\n[Step 2] Loading local WKT datasets and parsing geometries...")
matcher.conn.execute("""
    CREATE OR REPLACE TABLE driving_edges AS
    SELECT 
        edge_id::BIGINT AS edge_id,
        ST_GeomFromText(geometry) AS geometry
    FROM '../data/osm_edges.csv';
""")

matcher.conn.execute("""
    CREATE OR REPLACE TABLE vehicle_edges_directed AS
    SELECT 
        directed_id::BIGINT AS directed_id,
        original_edge_id::BIGINT AS original_edge_id,
        name,
        is_reverse::BOOLEAN AS is_reverse,
        ST_GeomFromText(geometry) AS geometry
    FROM '../data/sweden_edges.csv';
""")

# 3. Configure column mapping
matcher.configure_sources(
    source_a=TABLE_A, id_col_a=ID_A, geom_col_a=GEOM_A,
    source_b=TABLE_B, id_col_b=ID_B, geom_col_b=GEOM_B,
    utm_srid=UTM_SRID
)

# 4. Set matching thresholds
#matcher.set_parameters(max_distance=25.0, max_angle=30.0, min_overlap=0.50)
matcher.set_parameters(max_distance=25.0)

# 5. Tier 1: Candidate Generation
print("\n[Step 3] Finding candidate overlaps using DuckDB Spatial index...")
candidates = matcher.generate_candidate_pairs()
print(f"--> Found {len(candidates)} candidate pairs to evaluate.")

## Step 4: Run DTW Alignments & Reconcile

We evaluate shape similarity on the candidate list using Continuous Subsequence DTW and reconcile matching decisions in SQL.

In [ ]:
if not candidates.empty:
    # 5. Tier 2: DTW Shape matching
    print("[Step 4] Running 2D Subsequence DTW shape matching...")
    evaluated = matcher.compute_dtw_metrics(candidates)
    
    # 6. Tier 3: SQL Reconciliation & Split Road Detection
    print("\n[Step 5] Reconciling matches in SQL (Symmetric, Splits, Conflicts)...")
    results = matcher.reconcile_matches(evaluated)
    
    print("\nConflation Matches Summary:")
    print(results["match_type"].value_counts())
    
    # Display top 10 matches
    print("\nSample Matches Table:")
    display(results.head(10))
else:
    print("No candidates to match.")


## Step 5: Interactive Map Visualization

Let's see our matches visualized on an interactive dark-matter Leaflet map! We load the original geometries, overlay the matching decisions, and highlight match categories: 
* **Green**: Precise symmetric 1:1 matches.
* **Blue**: Coarse-to-fine 1:N splits.
* **Orange**: Directional conflicts or unidirectionally partial alignments.
* **Dashed Gray**: Unmatched OSM roads.
* **Dashed Red**: Unmatched Sweden NVDB roads.

In [ ]:
def visualize_matching_results(matcher, results_df):
    import folium
    import geopandas as gpd
    import pandas as pd
    from shapely.wkt import loads as load_wkt
    
    # 1. Fetch geometries from matcher's active connection
    print("Fetching road geometries from databases...")
    df_geom_a = matcher.conn.execute(f"SELECT {matcher.columns_a['id']} AS id_a, ST_AsText({matcher.columns_a['geom']}) AS wkt_a FROM {matcher.source_a}").df()
    df_geom_b = matcher.conn.execute(f"SELECT {matcher.columns_b['id']} AS id_b, name, is_reverse, ST_AsText({matcher.columns_b['geom']}) AS wkt_b FROM {matcher.source_b}").df()
    
    # Convert to GeoDataFrames
    gdf_a = gpd.GeoDataFrame(
        df_geom_a, 
        geometry=df_geom_a["wkt_a"].apply(load_wkt), 
        crs="EPSG:4326"
    ).drop(columns=["wkt_a"])
    
    gdf_b = gpd.GeoDataFrame(
        df_geom_b, 
        geometry=df_geom_b["wkt_b"].apply(load_wkt), 
        crs="EPSG:4326"
    ).drop(columns=["wkt_b"])
    
    # 2. Join matching results
    matched_a = gdf_a.merge(results_df, left_on="id_a", right_on="source_id", how="inner")
    matched_b = gdf_b.merge(results_df, left_on="id_b", right_on="dest_id", how="inner")
    
    # Find unmatched roads (NO_MATCH)
    unmatched_a = gdf_a[~gdf_a["id_a"].isin(results_df["source_id"])].copy()
    unmatched_b = gdf_b[~gdf_b["id_b"].isin(results_df["dest_id"])].copy()
    
    # 3. Create the folium map
    print("Generating Leaflet Map...")
    bounds = gdf_a.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="CartoDB dark_matter")
    
    # Define styles for match types
    colors = {
        "1:1_SYMMETRIC": "#10b981",       # Emerald Green
        "1:N_SPLIT": "#3b82f6",           # Vivid Blue
        "CONFLICT": "#f59e0b",            # Warm Orange
        "UNIDIRECTIONAL_PARTIAL": "#eab308" # Yellow
    }
    
    # Create FeatureGroups
    fg_symmetric = folium.FeatureGroup(name="🟢 1:1 Symmetric Matches (Green)", show=True)
    fg_splits = folium.FeatureGroup(name="🔵 1:N Split Matches (Blue)", show=True)
    fg_conflicts = folium.FeatureGroup(name="🟡 Conflicts & Partial Matches (Orange)", show=True)
    fg_unmatched_a = folium.FeatureGroup(name="⚪ Unmatched OSM Roads (Dashed Gray)", show=False)
    fg_unmatched_b = folium.FeatureGroup(name="🔴 Unmatched Sweden Roads (Dashed Red)", show=False)
    
    # Add matched OSM roads
    for _, row in matched_a.iterrows():
        m_type = row["match_type"]
        color = colors.get(m_type, "#94a3b8")
        
        # Tooltip details
        tooltip_html = f"""
        <div style='font-family: Arial, sans-serif; font-size: 12px;'>
            <strong>OSM Road ID:</strong> {row['id_a']}<br>
            <strong>Match Type:</strong> <span style='color: {color}; font-weight: bold;'>{m_type}</span><br>
            <strong>Avg DTW Distance:</strong> {row['dtw_distance']:.2f} m<br>
            <strong>Max DTW Distance:</strong> {row['max_dtw_distance']:.2f} m<br>
            <strong>Bearing Diff:</strong> {row['bearing_diff']:.1f}°<br>
            <strong>Overlap:</strong> {int(row['overlap_pct'])}%
        </div>
        """
        
        geojson = folium.GeoJson(
            row["geometry"].__geo_interface__,
            style_function=lambda x, color=color: {"color": color, "weight": 3.5, "opacity": 0.85},
            tooltip=folium.Tooltip(tooltip_html)
        )
        
        if m_type == "1:1_SYMMETRIC":
            geojson.add_to(fg_symmetric)
        elif m_type == "1:N_SPLIT":
            geojson.add_to(fg_splits)
        else:
            geojson.add_to(fg_conflicts)
            
    # Add unmatched OSM roads
    for _, row in unmatched_a.iterrows():
        tooltip_html = f"""
        <div style='font-family: Arial, sans-serif; font-size: 12px;'>
            <strong>Unmatched OSM Road ID:</strong> {row['id_a']}<br>
            <strong>Status:</strong> <span style='color: #94a3b8; font-weight: bold;'>NO_MATCH</span>
        </div>
        """
        folium.GeoJson(
            row["geometry"].__geo_interface__,
            style_function=lambda x: {"color": "#64748b", "weight": 1.8, "opacity": 0.6, "dashArray": "4, 6"},
            tooltip=folium.Tooltip(tooltip_html)
        ).add_to(fg_unmatched_a)
        
    # Add unmatched Sweden roads
    for _, row in unmatched_b.iterrows():
        dir_label = "Reverse" if row['is_reverse'] else "Forward"
        tooltip_html = f"""
        <div style='font-family: Arial, sans-serif; font-size: 12px;'>
            <strong>Unmatched Sweden Road ID:</strong> {row['id_b']}<br>
            <strong>Name:</strong> {row['name']}<br>
            <strong>Direction:</strong> {dir_label}<br>
            <strong>Status:</strong> <span style='color: #ef4444; font-weight: bold;'>NO_MATCH</span>
        </div>
        """
        folium.GeoJson(
            row["geometry"].__geo_interface__,
            style_function=lambda x: {"color": "#ef4444", "weight": 1.8, "opacity": 0.6, "dashArray": "4, 6"},
            tooltip=folium.Tooltip(tooltip_html)
        ).add_to(fg_unmatched_b)
        
    # Add groups to map
    fg_symmetric.add_to(m)
    fg_splits.add_to(m)
    fg_conflicts.add_to(m)
    fg_unmatched_a.add_to(m)
    fg_unmatched_b.add_to(m)
    
    # Layer control and fit bounds
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    
    print("✅ Interactive Map successfully generated!")
    return m

# Call the visualization function and show the map
if not results.empty:
    conflation_map = visualize_matching_results(matcher, results)

    # Save the interactive map to a standalone HTML file you can open in a browser
    html_out_path = "../data/conflation_map.html"
    conflation_map.save(html_out_path)
    print(f"✅ Saved interactive map to: {html_out_path}")

    display(conflation_map)
else:
    print("No results to visualize.")

## Step 6: Export Conflation Table

Write the final matching results back to a new table inside your primary Sundbyberg database, or save it to a flat GeoPackage/CSV file.

In [ ]:
if not candidates.empty:
    # Build the COMPLETE table: matched pairs + every OSM (Source A) segment that found
    # no match, appended as NO_MATCH rows (dest_id=None, metrics=NaN). This way the saved
    # file lists every OSM segment, matched or not -- not just the ones that matched.
    results_full = matcher._append_unmatched(results, matcher._get_all_ids_a())

    n_matched = (results_full["match_type"] != "NO_MATCH").sum()
    n_nomatch = (results_full["match_type"] == "NO_MATCH").sum()
    print(f"Rows: {n_matched} matched + {n_nomatch} NO_MATCH = {len(results_full)} total")

    # Register the combined DataFrame in DuckDB connection
    matcher.conn.register("final_matches", results_full)

    # Save the permanent table as a local CSV inside the project data directory
    csv_out_path = "../data/conflation_results.csv"
    matcher.conn.execute(f"COPY final_matches TO '{csv_out_path}' (HEADER, DELIMITER ',');")
    print(f"✅ Successfully saved conflation results to local CSV: {csv_out_path}")

    matcher.conn.close()
else:
    print("No results to export.")